<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/CustomAgentState.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 12.5 MB/s eta 0:00:00


In [3]:
import operator

from google.colab import userdata
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import AgentMiddleware, ToolCallRequest, ModelResponse, ModelRequest
from langchain.agents.middleware import after_agent, after_model, before_model, wrap_model_call
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.runtime import Runtime
from langgraph.types import Command
from typing import Annotated, Callable, List

api_key = userdata.get('OPENAI_API_KEY')

def print_conversation(conversation: List[AIMessage]):
    for message in conversation:
        message.pretty_print()

In [17]:
@before_model
def print_state_before_model(state: AgentState, runtime: Runtime):
    print("Before model")
    print(state)

@after_model
def print_state_after_model(state: AgentState, runtime: Runtime):
    print("After model")
    print(state)


class TrackUsageAgentState(AgentState):
    input_tokens: int
    cached_tokens: Annotated[int, operator.add] # reducer който казва + . prev + new
    output_tokens: int
    reasoning_tokens: Annotated[int, operator.add]

@after_model(state_schema = TrackUsageAgentState) # ще работи с custom agent state който наследява разширява дефолтния
def track_usage(state: TrackUsageAgentState, runtime: Runtime):
    last_message = state['messages'][-1]
    if last_message.usage_metadata is not None:
        input_tokens = last_message.usage_metadata.get("input_tokens", 0)
        cached_tokens = last_message.usage_metadata.get("input_token_details", {}).get("cache_read", 0)
        output_tokens = last_message.usage_metadata.get("output_tokens", 0)
        reasoning_tokens = last_message.usage_metadata.get("output_token_details", {}).get("reasoning", 0)

        #print(f"Input tokens: {input_tokens} ({cached_tokens} cached); Output tokens: {output_tokens} ({reasoning_tokens} reasoning)")
        # за да имплементираме промяна на state, да го мутираме -  трябва да върнем обект, при който обект дефинираме запис за всеки един input който трябва да бъде преобразуван
        return { "input_tokens": input_tokens, "cached_tokens": cached_tokens, "output_tokens": output_tokens, "reasoning_tokens": reasoning_tokens }

In [20]:
agent = create_agent(
    model = ChatOpenAI(model = "gpt-5-nano", api_key = api_key, reasoning_effort = "low"),
    #middleware=[print_state_before_model, print_state_after_model, TodoListMiddleware()]
    middleware=[track_usage, TodoListMiddleware()]
)

In [21]:
prove_irrational = agent.invoke(
    input = {
        "messages":[
            HumanMessage("Prove that the square roots of 2 and 3 are irrational. First plan your actions, prepare a list of TODO items and then start working them one by one. I want you to provide at least 2 independent proofs for both tasks")
        ]
    }
)

In [23]:
prove_irrational

{'messages': [HumanMessage(content='Prove that the square roots of 2 and 3 are irrational. First plan your actions, prepare a list of TODO items and then start working them one by one. I want you to provide at least 2 independent proofs for both tasks', additional_kwargs={}, response_metadata={}, id='9b1ffb88-6bfc-4b57-957c-f20c136f5f1f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1234, 'prompt_tokens': 1339, 'total_tokens': 2573, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1152}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EH38zaeoxMS84eVObCjcq93hEBmFj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03d27-1f1c-78f2-

In [22]:
print_conversation(prove_irrational['messages'])

================================ Human Message =================================

Prove that the square roots of 2 and 3 are irrational. First plan your actions, prepare a list of TODO items and then start working them one by one. I want you to provide at least 2 independent proofs for both tasks
================================== Ai Message ==================================
Tool Calls:
  write_todos (call_WWhPKdR5ZKDVFNhi9elH1GXU)
 Call ID: call_WWhPKdR5ZKDVFNhi9elH1GXU
  Args:
    todos: [{'content': 'Plan the approach: provide at least 2 independent proofs for sqrt(2) irrational and at least 2 independent proofs for sqrt(3) irrational. Use multiple proof techniques (parity, prime factorization, modular argument, infinite descent). Create clear, self-contained proofs for each claim.', 'status': 'in_progress'}, {'content': "Proof 1 for sqrt(2) using Euclid's parity argument (if sqrt(2)=a/b in lowest terms then a and b have same parity leading to contradiction).", 'status': 'pending'}